# **RepuSense · Radar de Reputación en Crisis**

Este notebook construye la **base de leads** del proyecto: negocios (hoteles y restaurantes) o cualquier otro giro de negocio que quieran investigar con *mala reputación en Google Maps*, con sus datos de contacto y las quejas de sus clientes, listos para el equipo comercial.

## **¿Qué es el web scraping?**

Es **extraer datos de páginas web de forma automática**. En lugar de que una persona abra Google Maps, copie el nombre, el rating y el teléfono de cada negocio, un programa lo hace por ella: abre la página, "lee" su código HTML y guarda los datos en una tabla.

Este código usa tres herramientas, cada una con un trabajo distinto:

| Herramienta | ¿Qué hace? | Para qué la usamos |
|---|---|---|
| **Selenium** | Ir en "persona" y recorrer el lugar | Controla un Chrome real: Google Maps solo muestra sus datos cuando alguien navega (scroll, clics) |
| **requests** | Pedir el menú impreso por correo | Descarga la página web de cada negocio, sin abrir el navegador (rápido) |
| **BeautifulSoup** | La "persona" que encuentra lo que busca en el menú | Recorre el HTML descargado y encuentra correos, teléfonos y enlaces |

## Mapa del notebook

| Paso | Celda | Qué hace |
|---|---|---|
| 1–2 | Librerías | Carga las herramientas |
| 3 | **Parámetros** | Ciudades, rubros y reglas del cliente. **Es la única celda que normalmente se modifica** |
| 4 | Archivos | Nombres de los archivos de salida |
| 5–6 | Funciones de apoyo | Pausas, lectura de números, teléfonos de cualquier país |
| 7 | **Fase 1** | Busca en Maps cada rubro × ciudad y guarda **todos** los negocios |
| 8 | **Fase 2.1** | Abre la ficha de cada candidato: teléfono, web, histograma de estrellas, reseñas |
| 9 | **Fase 2.2** | Visita la web del negocio para buscar correo, celular y redes sociales |
| 10 | **Ejecución** | Corre las fases en orden y clasifica los leads (Fase 3) |
| 11 | Revisión | Resume los resultados (se puede ejecutar sola) |

```
Fase 1: Google Maps (lista)  →  negocios_11.csv   (todos, con fecha: historial)
        │  filtro del cliente: 1.5 ≤ rating ≤ 3.5 y > 10 reseñas
Fase 2: ficha + web del negocio  →  fichas_11.csv + resenas_11.csv
        │
Fase 3: clasificación (sin volver a Maps)  →  leads_11.csv   (Alta / Media)
```

## Cómo ejecutarlo

1. Abrir la carpeta del proyecto en VS Code y seleccionar el kernel `.venv` (arriba a la derecha).
2. Revisar la **celda 3** (ciudades, rubros, reglas).
   - En esa celda se elige el bloque de países de esta sesión: `BLOQUE_ACTIVO = 1` … `6`.
3. Pulsar **Restart** y luego **Run All**.
4. No usar ni cerrar la ventana de Chrome que se abre. Una ejecución completa puede tardar varias horas: la Fase 1 hace una búsqueda por cada rubro × ciudad, y la Fase 2 visita hasta `LOTE_FICHAS` fichas.
5. Si se interrumpe, volver a ejecutar: **todo es reanudable** (lo ya guardado se salta).
6. En la siguiente sesión (idealmente otro día), cambiar `BLOQUE_ACTIVO` al bloque siguiente. Los resultados se acumulan en los mismos archivos.

## Glosario

| Término | Significado |
|---|---|
| **Lead** | Negocio que cumple la regla del cliente para nuestro caso (1.5 – 3.5 y > 10 reseñas): un cliente potencial de la agencia |
| **place_id** | Identificador único que Google asigna a cada negocio. Es la **llave** para evitar duplicados |
| **Ficha** | La página de un negocio en Google Maps (teléfono, web, reseñas) |
| **Histograma** | Cuántas reseñas de 5, 4, 3, 2 y 1 estrella tiene un negocio |
| **pct_negativas** | Proporción de reseñas de 1-2 estrellas. Mide la crisis mejor que el promedio |
| **Percentil** | Posición frente a su competencia (mismo rubro, país y ciudad). 0.05 = entre el 5% peor |
| **Snapshot** | El registro de un negocio en una fecha. Varios snapshots forman su historial |
| **DOM / HTML** | La estructura de una página web. F12 en el navegador la muestra |
| **Bloque** | Grupo de 2-3 países que se ejecuta en una sesión, para no saturar a Google |
| **E.164** | Formato internacional de teléfono: `+593991234567` |

## ¿Por qué Google muestra solo unos 120 resultados por búsqueda?

El límite es **por búsqueda**, no por país ni por día. Google Maps carga la lista de 20 en 20 resultados mientras se hace scroll, y después de unas 6 "páginas" (unos 120 negocios) muestra *"Has llegado al final de la lista"*. Google lo limita porque casi ninguna persona revisa más allá, porque reduce el costo de sus servidores y porque dificulta la extracción masiva.

Además, la lista está ordenada por *relevancia*, que favorece a los negocios con más y mejores reseñas. En una ciudad grande hay miles de restaurantes, pero solo se ven los 120 "mejores" puntuados o calificados, casi todos bien calificados. Por eso:

- **Más ciudades = más universo**: cada ciudad es una búsqueda nueva con sus propios 120 resultados.
- Las ciudades no comunes suelen aportar más leads por búsqueda que las capitales o ciudades grandes.
- Para profundizar en una capital, se puede buscar por *barrios* ("HOTELES en Palermo, Buenos Aires").

## ¿Por qué ejecutar por bloques?

El riesgo de bloqueo depende de **cuántas acciones seguidas** hace el programa en una sesión, no del total. Con los nuevos ajustes este código esta construido para ejecutar la búsqueda en 17 países × 12 ciudades × 2 rubros = **408 búsquedas** esto se traduce en un aprox de más de 10 horas de búsqueda un patrón claramente automático con riesgo de bloqueo.Para este proyecto se divide la consulta por **6 bloques** de unas 72 búsquedas, con pausas aleatorias y un descanso de 1-2 minutos cada 12 búsquedas, cada sesión se parece a un uso intenso pero normal.

## Documentación

- Selenium: https://www.selenium.dev/documentation/ · API Python: https://www.selenium.dev/selenium/docs/api/py/index.html
- Requests: https://requests.readthedocs.io/en/latest/
- Beautiful Soup (español): https://www.crummy.com/software/BeautifulSoup/bs4/doc.es/
- phonenumbers: https://github.com/daviddrysdale/python-phonenumbers
- Lógica de contactos adaptada de [jsonifyco/gmaps-scraper](https://github.com/jsonifyco/gmaps-scraper) (`enrich/contacts.py`, licencia MIT). No usa su servicio de pago ni proxies.

**Límite de Google Maps sin iniciar sesión:** muestra el histograma y unas 5 reseñas "más relevantes"; el botón *Ordenar* pide iniciar sesión. No iniciamos sesión (términos de uso de Google): el histograma es la señal principal.

## Paso 1 · Librerías base

Una **librería** es código ya escrito que importamos para no programarlo desde cero.

| Librería | Para qué |
|---|---|
| `json`, `re`, `random`, `time`, `datetime`, `pathlib`, `urllib` | Vienen con Python: texto estructurado, patrones de texto (regex), pausas, fechas, rutas de archivos, URLs |
| `pandas` | Tablas (como Excel dentro de Python): leer, unir, filtrar y guardar CSV |
| `requests` | Descargar páginas web sin abrir el navegador |
| `selenium` | Controlar Chrome: abrir páginas, esperar, hacer clic y scroll |


In [2]:
import json
import random
import re
import time
from datetime import date
from pathlib import Path
from urllib.parse import quote_plus, unquote, urljoin, urlparse

import pandas as pd
import requests

from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

## Paso 2 · Librerías adicionales

`%pip install` instala las librerías **en el mismo Python que usa este notebook** (si ya están, no hace nada). Si aparece *"you may need to restart the kernel"*, pulsar **Restart** y volver a **Run All**.

Ojo: el paquete se instala como `beautifulsoup4`, pero se importa como `bs4`.


In [3]:
# Librerías adicionales. %pip instala en el MISMO Python que usa el kernel (si ya están instaladas, no hace nada).
# Nombre para instalar: beautifulsoup4 | nombre para importar: bs4

# El código abajo le retiras el símbolo # para activarlo y ejecutarlo, solamente si no tienes instalado bs4
# %pip install -q beautifulsoup4 phonenumbers

# leer el HTML de la web del negocio (sin abrir Chrome)
from bs4 import BeautifulSoup

# validar y clasificar teléfonos de cualquier país
import phonenumbers

## Paso 3 · Parámetros
## En esta celda se puede personalizar la búsqueda de acuerdo a los requerimientos del proyecto o investigación
- *`CIUDADES`*: pares `("ciudad", "país")`. Cada par genera una búsqueda por rubro: `"HOTELES en Manta, Ecuador"`.
- *`DESCRIPCION`*: los rubros. Búsquedas totales = rubros × ciudades.
- *Regla del cliente*: `RATING_MIN` ≤ rating ≤ `RATING_MAX` y más de `MIN_RESENAS` reseñas. Solo estos negocios se visitan en la Fase 2.
- *Prioridad*: dentro de los leads, **Alta** si cumple al menos una señal fuerte (muchas reseñas negativas, rating muy bajo o caída del rating); si no, **Media**.
- *`LOTE_FICHAS`*: cuántas fichas se visitan por ejecución. Más alto = termina antes, pero con más riesgo de que Google limite la vista (en la v09, con 300, las últimas 50 fichas salieron sin reseñas).

> Con nueva regla 1.5–3.5, en la v09 solo el **3%** de los negocios calificó (140 de 4.750). Para superar 200 leads hay que *agregar ciudades* (de preferencia pueblos pequeños y hoteles, donde aparecen más negocios mal calificados).

**Ejecución por bloques (v11)**
- *`CIUDADES_TODAS`*: el universo completo (17 países × 12 ciudades). No se modifica en cada sesión.
- *`BLOQUES`*: agrupa los países de a 2-3. Se puede editar para agrupar distinto (nombres de país escritos igual que en la lista, con tildes).
- *`BLOQUE_ACTIVO`*: **el único número que se cambia en cada sesión** (1 a 6). La línea `CIUDADES = [c for c in CIUDADES_TODAS if c[1] in PAISES_ACTIVOS]` arma la lista de la sesión: conserva solo los pares cuyo país (`c[1]`) está en el bloque activo.
- *`DESCANSO_CADA` / `DESCANSO_SEG`*: cada 12 búsquedas, una pausa de 1-2 minutos.

| Bloque | Países | Búsquedas |
|---|---|---|
| 1 | Ecuador, Colombia, Perú | 72 |
| 2 | Argentina, Chile, Uruguay | 72 |
| 3 | México, Guatemala, El Salvador | 72 |
| 4 | Brasil, Paraguay, Bolivia | 72 |
| 5 | Venezuela, Panamá, Costa Rica | 72 |
| 6 | Honduras, Nicaragua | 48 |

Al ejecutar la celda aparece la confirmación, por ejemplo: `Bloque 2: Argentina, Chile, Uruguay → 36 ciudades × 2 rubros = 72 búsquedas`.


In [4]:
# Configuración de la búsqueda

# Cada ciudad va con su país: ("ciudad", "país"). El país debe existir en CODIGO_PAIS (celda de teléfonos).
# UNIVERSO DE BÚSQUEDA: 17 países × 12 ciudades (las más pobladas + destinos turísticos)

CIUDADES_TODAS = [
    # MÉXICO
    ("Ciudad de México", "México"), ("Guadalajara", "México"), 
]

# EJECUCIÓN POR BLOQUES (para reducir el riesgo de bloqueo)
# Cada bloque = 3 países = 36 ciudades × 2 rubros = 72 búsquedas esto puede tomar entre 1 a 2 horas.
# Ejecutar un (1) bloque por sesión, idealmente en días distintos. 
# Todo se acumula en los mismos archivos.

BLOQUES = {
    # 1: ["Ecuador", "Colombia", "Perú"],
    # 2: ["Argentina", "Chile", "Uruguay"],
    1: ["México"],
    # 4: ["Brasil", "Paraguay", "Bolivia"],
    # 5: ["Venezuela", "Panamá", "Costa Rica"],
    # 6: ["Honduras", "Nicaragua"],
}

# Está linea de código es muy importante, ya que, en cada ejecución se debe seleccionar el bloque [1;2;3;4;5;6]
BLOQUE_ACTIVO = 1

PAISES_ACTIVOS = BLOQUES[BLOQUE_ACTIVO]
CIUDADES = [c for c in CIUDADES_TODAS if c[1] in PAISES_ACTIVOS]

DESCRIPCION = ["HOTELES", "RESTAURANTES"]

# Descanso largo dentro de la Fase 1: cada cierto número de búsquedas, una pausa de 1 a 2 minutos
DESCANSO_CADA = 12         # búsquedas entre descansos
DESCANSO_SEG = (60, 120)   # duración del descanso, en segundos (mínimo, máximo)

print(f"Bloque {BLOQUE_ACTIVO}: {', '.join(PAISES_ACTIVOS)} → {len(CIUDADES)} ciudades × "
      f"{len(DESCRIPCION)} rubros = {len(CIUDADES) * len(DESCRIPCION)} búsquedas")

# REGLA DEL CLIENTE: qué negocio es un lead

# Rating mínimo 
RATING_MIN = 1.5

# Rating máximo
RATING_MAX = 3.5

# Más de 10 reseñas
MIN_RESENAS = 10

# PRIORIDAD dentro de los leads (Alta = llamar primero; el resto es Media) esta categorización es para el área comercial.
PCT_NEG_ALTA = 0.40       # Alta si el 40% o más de sus reseñas son de 1-2 estrellas...
RATING_ALTA = 3.0         # o si su rating es 3.0 o menos...
CAIDA_ALERTA = 0.2        # o si su rating cayó 0.2 o más desde la búsqueda anterior
UMBRAL_TENDENCIA = 0.1    # variación mínima para decir que "empeora" o "mejora"


# VOLUMEN Y RIESGO DE BLOQUEO
LOTE_FICHAS = 120         # fichas visitadas por ejecución.
REINTENTAR_FICHAS = True  # volver a visitar fichas que quedaron sin histograma o sin reseñas
MAX_INTENTOS = 2          # visitas máximas por ficha + 1 reintento


# RESEÑAS (sin iniciar sesión, Google muestra el histograma y unas 5 reseñas)
RATING_NEGATIVA = 2       # 1 y 2 estrellas son quejas claras
MESES_RECIENTE = 12       # recoge información de los 12 meses o menos

# Configuración de la búsqueda de correo y celular en la web del negocio

# páginas de contacto a revisar por web, además de la página de inicio
MAX_PAGINAS_CONTACTO = 3

# solo si la web no tiene enlaces de contacto
PAGINAS_CONTACTO = ["/contacto", "/contactanos", "/contact"]

# Segunda opción: abrir con Chrome las webs donde requests no encontró correo
USAR_SELENIUM_WEB = True   

# páginas extra por web como segunda opción
MAX_PAGINAS_SELENIUM = 2
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/130.0 Safari/537.36"}

Bloque 1: México → 2 ciudades × 2 rubros = 4 búsquedas


## Paso 4 · Archivos de salida (carpeta `data`)

| Archivo | Una fila por… | Contenido |
|---|---|---|
| `negocios_11.csv` | negocio **y fecha** | Todo lo encontrado en Maps (nombre, rating, reseñas, URL). Es el historial |
| `fichas_11.csv` | negocio visitado | Contacto, histograma, temas, web, redes |
| `resenas_11.csv` | reseña | Texto, estrellas, fecha y fuente de cada reseña visible (insumo de la IA) |
| `leads_11.csv` | lead | Leads clasificados (Alta / Media), con las columnas clave primero |

**No abrir y guardar estos CSV con Excel**: cambia separadores y decimales y daña el archivo. Para analizarlos, hacer una copia.


In [5]:
# Configuración del almacenamiento de la data

CARPETA = Path("data")
CARPETA.mkdir(exist_ok=True)
SALIDA_NEGOCIOS = CARPETA / "negocios_11.csv"
SALIDA_FICHAS = CARPETA / "fichas_11.csv"     # una fila por negocio visitado
SALIDA_RESENAS = CARPETA / "resenas_11.csv"   # una fila por reseña visible
SALIDA_LEADS = CARPETA / "leads_11.csv"       # leads clasificados (Alta / Media)
HOY = str(date.today())

# Columnas que deben leerse como texto (si no, pandas convierte +593991234567 en número y lo daña)
COLUMNAS_TEXTO = {"place_id": str, "telefono": str, "celular": str, "celulares_web": str,
                  "rating_anterior": float, "variacion_rating": float}

## Paso 5 · Funciones de apoyo

Una **función** (`def`) es un bloque de código con nombre que se escribe una vez y se usa muchas veces. Aquí: pausas aleatorias (para parecer una persona), extraer el `place_id` de la URL, convertir `"4,3 estrellas"` en `4.3`, leer un CSV si existe y abrir Chrome.


In [6]:
# FUNCIONES

def pausa(minimo, maximo):
    """Pausa aleatoria para imitar la búsqueda de una persona"""
    time.sleep(random.uniform(minimo, maximo))

def extraer_place_id(url):
    """Identificador único de Google dentro de la URL: """
    m = re.search(r"!1s(0x[0-9a-f]+:0x[0-9a-f]+)", url or "")
    return m.group(1) if m else None

def primer_numero(texto):
    """4,3 estrellas ==> 4.3 | 1 estrella ==> 1.0 """
    m = re.search(r"\d+(?:[.,]\d+)?", texto or "")
    return float(m.group().replace(",", ".")) if m else None

def texto_o_none(elemento, selector):
    """Texto de un sub-elemento, o None si no existe"""
    try:
        return elemento.find_element(By.CSS_SELECTOR, selector).text
    except NoSuchElementException:
        return None

def cargar_csv(ruta):
    if ruta.exists() and ruta.stat().st_size > 0:
        return pd.read_csv(ruta, dtype=COLUMNAS_TEXTO)
    return pd.DataFrame()

def iniciar_driver():
    # No usamos headless: Google detecta más fácil un navegador invisible
    options = webdriver.ChromeOptions()
    options.add_argument("--lang=es")
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()
    return driver

## Paso 6 · Teléfonos de cualquier país

La librería **phonenumbers** conoce las reglas de numeración de cada país. Convierte `099 123 4567` (Ecuador) en `+593991234567` y clasifica el número como `celular`, `fijo` o `celular_o_fijo` (México y otros no distinguen celular de fijo por el número).


In [7]:
# TELÉFONOS (multipaís, con la librería phonenumbers) Y CORREOS OCULTOS POR CLOUDFLARE
# Todos los números se guardan en formato internacional E.164: +593991234567 (Ecuador), +525512345678 (México)

CODIGO_PAIS = {"Ecuador": "EC", "Colombia": "CO", "México": "MX", "Mexico": "MX", "Argentina": "AR",
               "Perú": "PE", "Peru": "PE", "Chile": "CL", "Bolivia": "BO", "Uruguay": "UY", "Paraguay": "PY",
               "Venezuela": "VE", "Panamá": "PA", "Panama": "PA", "Costa Rica": "CR", "Guatemala": "GT",
               "El Salvador": "SV", "Honduras": "HN", "Nicaragua": "NI", "República Dominicana": "DO",
               "Cuba": "CU", "Brasil": "BR"}


def region_de(pais):
    """'México' ==> 'MX'. Si el país no está en la tabla, se asume Ecuador."""
    return CODIGO_PAIS.get(str(pais).strip(), "EC")


def normalizar_telefono(numero, pais="Ecuador"):
    """'099 123 4567' (Ecuador) ==> '+593991234567' | '55 1234 5678' (México) ==> '+525512345678'.
    Devuelve None si no es un número válido para ese país."""
    texto = str(numero or "").strip()
    if not texto:
        return None
    # 1.º se interpreta como número del país; 2.º, si son solo dígitos, como internacional sin '+' (wa.me/593...)
    for candidato in (texto, "+" + texto if texto.isdigit() else None):
        if not candidato:
            continue
        try:
            n = phonenumbers.parse(candidato, region_de(pais))
        except phonenumbers.NumberParseException:
            continue
        if phonenumbers.is_valid_number(n):
            return phonenumbers.format_number(n, phonenumbers.PhoneNumberFormat.E164)
    return None


TIPOS_TELEFONO = {phonenumbers.PhoneNumberType.MOBILE: "celular",
                  phonenumbers.PhoneNumberType.FIXED_LINE: "fijo",
                  phonenumbers.PhoneNumberType.FIXED_LINE_OR_MOBILE: "celular_o_fijo"}   # México, Argentina...


def tipo_telefono(numero_e164):
    """'+593991234567' ==> 'celular' | '+59322345678' ==> 'fijo' | '+525512345678' ==> 'celular_o_fijo'"""
    if not numero_e164:
        return None
    try:
        n = phonenumbers.parse(numero_e164)
    except phonenumbers.NumberParseException:
        return None
    return TIPOS_TELEFONO.get(phonenumbers.number_type(n), "otro")


def es_celular(numero_e164):
    """Sirve para WhatsApp: celular seguro, o 'celular_o_fijo' en países que no los distinguen."""
    return tipo_telefono(numero_e164) in ("celular", "celular_o_fijo")


def telefonos_en_texto(texto, pais="Ecuador"):
    """Encuentra teléfonos válidos dentro de un texto largo, con las reglas del país."""
    return [phonenumbers.format_number(m.number, phonenumbers.PhoneNumberFormat.E164)
            for m in phonenumbers.PhoneNumberMatcher(texto or "", region_de(pais))]


def decodificar_cfemail(codigo):
    """Correos que Cloudflare oculta en la web (atributo data-cfemail)"""
    clave = int(codigo[:2], 16)
    return "".join(chr(int(codigo[i:i + 2], 16) ^ clave) for i in range(2, len(codigo), 2))


## Paso 7 · Fase 1: lista de resultados de Google Maps

Cómo "lee" Maps el programa:

1. Abre la URL de búsqueda (`/maps/search/HOTELES en Manta, Ecuador`).
2. Espera a que aparezca el panel de resultados (`div[role="feed"]`).
3. Hace **scroll dentro del panel**: Maps carga los resultados de a poco (*lazy loading*), hasta unos 120 por búsqueda.
4. De cada tarjeta (`div[role="article"]`) lee nombre, rating, n.º de reseñas y URL.

Usamos atributos de accesibilidad (`role`, `aria-label`) y no las clases CSS de Google (`hfpxzc`…), porque las clases cambian sin aviso y los atributos de accesibilidad son estables.


In [8]:
# 1. Lista de resultados

def scroll_feed(driver, feed, max_rondas=30):
    anterior, sin_cambios = 0, 0
    for _ in range(max_rondas):
        tarjetas = feed.find_elements(By.CSS_SELECTOR, 'div[role="article"]')
        if tarjetas:   # llevar la última tarjeta a la vista obliga a Maps a cargar más resultados
            driver.execute_script("arguments[0].scrollIntoView();", tarjetas[-1])
        driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", feed)
        pausa(1.5, 3.5)
        n = len(feed.find_elements(By.CSS_SELECTOR, 'div[role="article"]'))
        if "final de la lista" in feed.text:
            break
        sin_cambios = sin_cambios + 1 if n == anterior else 0
        if sin_cambios >= 3:
            break
        anterior = n


def extraer_tarjeta(t):
    enlace = t.find_element(By.CSS_SELECTOR, "a[aria-label]")
    url = enlace.get_attribute("href")

    rating, n_resenas = None, None
    for e in t.find_elements(By.CSS_SELECTOR, 'span[role="img"]'):
        label = e.get_attribute("aria-label") or ""
        if "estrella" in label.lower():
            nums = re.findall(r"\d[\d.,]*", label)
            rating = float(nums[0].replace(",", "."))
            if len(nums) > 1:
                n_resenas = int(re.sub(r"[.,]", "", nums[1]))
            break

    return {
        # Si no hay ID en la URL usamos la URL: evita que las filas sin ID se fusionen en una sola
        "place_id": extraer_place_id(url) or url,
        "nombre": enlace.get_attribute("aria-label"),
        "rating": rating,
        "n_resenas": n_resenas,
        "url": url,
    }


def extraer_lista(driver, wait, descripcion, ciudad, pais):
    busqueda = f"{descripcion} en {ciudad}, {pais}"
    driver.get(f"https://www.google.com/maps/search/{quote_plus(busqueda)}?hl=es")
    try:
        feed = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="feed"]')))
    except TimeoutException:
        print(f"    Sin lista de resultados para {ciudad}.")
        return []

    scroll_feed(driver, feed)

    filas = []
    for t in feed.find_elements(By.CSS_SELECTOR, 'div[role="article"]'):
        try:
            fila = extraer_tarjeta(t)
            fila.update(descripcion=descripcion, ciudad=ciudad, pais=pais, fecha_extraccion=HOY)
            filas.append(fila)
        except Exception:
            continue
    return filas

## Paso 8 · Fase 2.1: ficha del negocio y reseñas

En la ficha de cada candidato se lee:

- **Contacto**: teléfono (`data-item-id="phone:tel:…"`), web y dirección.
- **Histograma**: cuántas reseñas de cada estrella → `pct_negativas` = (1★ + 2★) / total.
- **Temas**: palabras que Google detecta en las reseñas, con cuántas las mencionan (`"limpieza: 5; precio: 3"`).
- **Reseñas visibles** (unas 5): estrellas, fecha, fuente (Google, Tripadvisor…) y texto.

No se pulsa "Ordenar": sin sesión iniciada, Google abre una ventana de inicio de sesión.


In [9]:
# 2. Detalles y reseñas

def extraer_detalle(driver, wait, url, pais="Ecuador"):
    """Abre la ficha del negocio por URL (no por clic) y saca sus datos de contacto."""
    driver.get(url)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    pausa(2, 4)

    detalle = {"telefono": None, "tipo_telefono": None, "web": None, "direccion": None}
    try:
        # data-item-id = "phone:tel:022345678" -> nos quedamos con lo que va tras "tel:"
        tel = driver.find_element(By.CSS_SELECTOR, 'button[data-item-id^="phone"]')
        detalle["telefono"] = normalizar_telefono(tel.get_attribute("data-item-id").split("tel:")[-1], pais)
        detalle["tipo_telefono"] = tipo_telefono(detalle["telefono"])   # celular / fijo / celular_o_fijo / otro
    except NoSuchElementException:
        pass
    try:
        web = driver.find_element(By.CSS_SELECTOR, 'a[data-item-id="authority"]')
        detalle["web"] = web.get_attribute("href")
    except NoSuchElementException:
        pass
    try:
        dire = driver.find_element(By.CSS_SELECTOR, 'button[data-item-id="address"]')
        detalle["direccion"] = (dire.get_attribute("aria-label") or "").split(": ", 1)[-1]
    except NoSuchElementException:
        pass
    return detalle


# Reseñas SIN iniciar sesión: Google muestra el resumen (histograma y temas) y unas 5 reseñas.
# NO pulsamos "Ordenar": abre la ventana "Inicia sesión" y no ordena nada.
XPATH_PESTANA = '//button[@role="tab"][contains(@aria-label,"Reseñas") or contains(@aria-label,"Opiniones")]'
FECHA_RE = re.compile(r"hace\s+(?:un|una|\d+)\s+(?:minuto|hora|día|semana|mes|año)\w*", re.I)   # "Hace 7 años"
FUENTES_CONOCIDAS = r"Google|Tripadvisor|Booking(?:\.com)?|Expedia|Hotels\.com|Priceline|Trip\.com|Agoda|OYO|Despegar|Facebook"
FUENTE_RE = re.compile(r"hace[^\n]*?\ben\b\s*\n?\s*(" + FUENTES_CONOCIDAS + r")\b", re.I)   # si no es conocida, queda vacía                               # "... en Tripadvisor"


def cerrar_dialogo(driver):
    """Cierra la ventana 'Inicia sesión' si apareció."""
    for boton in driver.find_elements(By.XPATH, '//div[@role="dialog"]//button[contains(., "Cerrar")]'):
        try:
            boton.click()
            pausa(0.5, 1)
        except Exception:
            pass


def abrir_resenas(driver, wait):
    """Abre la pestaña de reseñas. Devuelve True si pudo hacer clic."""
    try:
        wait.until(EC.element_to_be_clickable((By.XPATH, XPATH_PESTANA))).click()
        pausa(2, 3)
        cerrar_dialogo(driver)
        return True
    except TimeoutException:
        return False


def leer_histograma(driver):
    """{5: 12, 4: 3, 3: 0, 2: 2, 1: 5} desde los aria-label del resumen ('5 estrellas, 12 reseñas')."""
    hist = {}
    for e in driver.find_elements(By.CSS_SELECTOR, '[aria-label*="estrella"]'):
        m = re.match(r"\s*([1-5])\s+estrellas?,?\s+([\d.,]+)\s+(?:reseña|opinion|opinión)",
                     e.get_attribute("aria-label") or "", re.I)
        if m:
            hist[int(m.group(1))] = int(re.sub(r"[.,]", "", m.group(2)))
    return hist


def leer_temas(driver):
    """Temas que Google detecta en las reseñas, con cuántas los mencionan: 'playa: 3; dinero: 2'."""
    temas = {}
    for e in driver.find_elements(By.CSS_SELECTOR, '[role="radio"]'):
        partes = [p.strip() for p in (e.text or "").split("\n") if p.strip()]
        if len(partes) == 2 and partes[1].isdigit():
            temas[partes[0]] = partes[1]
    return "; ".join(f"{t}: {n}" for t, n in temas.items())


def meses_desde(fecha_texto):
    """'hace 2 meses' ==> 2 | 'Hace un año' ==> 12 | 'hace 3 semanas' ==> 0 | sin fecha ==> None"""
    if not fecha_texto:
        return None
    t = fecha_texto.lower()
    n = re.search(r"\d+", t)
    n = int(n.group()) if n else 1            # "un mes", "un año" = 1
    if "año" in t:
        return 12 * n
    if "mes" in t:
        return n
    return 0                                   # minutos, horas, días o semanas


def rating_de_resena(bloque):
    """Restaurantes: estrellas en aria-label ('1 estrella'). Hoteles: texto '1/5'."""
    for e in bloque.find_elements(By.CSS_SELECTOR, 'span[role="img"][aria-label]'):
        label = e.get_attribute("aria-label") or ""
        if "estrella" in label.lower():
            return primer_numero(label)
    m = re.search(r"\b([1-5])\s*/\s*5\b", bloque.text)
    return float(m.group(1)) if m else None


def bajar_a_resenas(driver):
    """Hace scroll hasta la lista de reseñas para que Maps las cargue."""
    ancla = driver.find_elements(By.CSS_SELECTOR, '[role="radio"]') or driver.find_elements(By.XPATH, XPATH_PESTANA)
    if ancla:
        driver.execute_script("arguments[0].scrollIntoView();", ancla[-1])
        pausa(1.5, 2.5)
    for _ in range(2):
        bloques = driver.find_elements(By.CSS_SELECTOR, "div[data-review-id]")
        if not bloques:
            break
        driver.execute_script("arguments[0].scrollIntoView();", bloques[-1])
        pausa(1.5, 2.5)


def leer_resenas_visibles(driver):
    """Lee las reseñas que Google muestra sin iniciar sesión (unas 5)."""
    bajar_a_resenas(driver)
    resenas = {}
    for bloque in driver.find_elements(By.CSS_SELECTOR, "div[data-review-id]"):
        rid = bloque.get_attribute("data-review-id")
        if rid in resenas:                     # el atributo se repite en divs anidados
            continue
        rating = rating_de_resena(bloque)
        if rating is None:
            continue
        for boton in bloque.find_elements(By.XPATH, './/button[normalize-space(.)="Más" or @aria-label="Ver más"]'):
            try:
                boton.click()                  # expandir textos largos
            except Exception:
                pass
        cerrar_dialogo(driver)
        lineas = [l.strip() for l in bloque.text.split("\n") if l.strip()]
        fecha = FECHA_RE.search(bloque.text)
        fuente = FUENTE_RE.search(bloque.text)
        resenas[rid] = {
            "rating": int(rating),
            "autor": texto_o_none(bloque, ".d4r55") or (lineas[0] if lineas else None),
            "fecha": fecha.group(0) if fecha else None,
            "meses": meses_desde(fecha.group(0)) if fecha else None,
            "fuente": fuente.group(1) if fuente else None,           # Google, Tripadvisor...
            "texto": texto_o_none(bloque, ".wiI7pd") or (max(lineas, key=len) if lineas else ""),
            "respondida": texto_o_none(bloque, ".CDe7pd .wiI7pd") is not None,
        }
    return list(resenas.values())


def resumen_resenas(driver, wait):
    """Devuelve (resumen para la ficha, lista de reseñas visibles)."""
    resumen = {**{f"hist_{k}": None for k in range(5, 0, -1)},
               "pct_negativas": None, "temas": "", "resenas_visibles": 0,
               "negativas_visibles": 0, "negativas_visibles_recientes": 0}
    if not abrir_resenas(driver, wait):
        return resumen, []
    hist = leer_histograma(driver)
    temas = leer_temas(driver)
    visibles = leer_resenas_visibles(driver)

    total = sum(hist.values())
    negativas = [r for r in visibles if r["rating"] <= RATING_NEGATIVA]
    resumen.update({f"hist_{k}": hist.get(k) for k in range(5, 0, -1)})
    resumen.update(
        pct_negativas=round((hist.get(1, 0) + hist.get(2, 0)) / total, 3) if total else None,
        temas=temas,
        resenas_visibles=len(visibles),
        negativas_visibles=len(negativas),
        negativas_visibles_recientes=sum(1 for r in negativas
                                         if r["meses"] is not None and r["meses"] <= MESES_RECIENTE))
    return resumen, visibles


## Paso 9 · Fase 2.2: contactos desde la web del negocio

Google Maps **no publica correos**. Se buscan en la web del negocio:

1. **requests + BeautifulSoup** descargan la página de inicio y siguen los enlaces reales de "Contacto", "Reservas" o "Nosotros".
2. Se detectan correos normales, en enlaces `mailto:`, ocultos para robots (`info [at] hotel [dot] com`) y protegidos por Cloudflare.
3. **Plan B**: si no hubo correo (webs hechas con JavaScript, como Wix), se abre la web con Selenium, que sí ejecuta JavaScript.
4. Se elige el mejor correo: primero el del mismo dominio que la web, luego gmail/hotmail; se descartan los técnicos (`noreply@`, `support.com`).

`tipo_web` indica qué era la "web" de la ficha: `propia`, `plataforma` (Booking, OYO: se salta), `link_en_bio` (Linktree: se lee), `red_social` o `whatsapp`.


In [10]:
# 3. Correo, celular y redes sociales desde la web del negocio
# Google Maps no publica correos: hay que buscarlos en la página web que aparece en la ficha.
# Lógica adaptada de jsonifyco/gmaps-scraper (src/gmaps_scraper/enrich/contacts.py), licencia MIT, (c) 2026 bro.

# Correos
EMAIL_COMPLETO = re.compile(r"[a-z0-9][a-z0-9._%+-]{0,63}@(?:[a-z0-9-]{1,63}\.)+[a-z]{2,24}", re.I)
EMAIL_EN_TEXTO = re.compile(r"(?<![\w.+-])([a-z0-9][a-z0-9._%+-]{0,63}@(?:[a-z0-9-]{1,63}\.)+[a-z]{2,24})(?![\w-])", re.I)
# Correos escritos para esquivar robots: "info [at] hotel [dot] com", "info arroba hotel punto com"
EMAIL_OCULTO = re.compile(
    r"([a-z0-9][a-z0-9._%+-]{0,63})\s*(?:\[at\]|\(at\)|\{at\}|\[@\]|\(@\)|\[arroba\]|\(arroba\)|\s+arroba\s+)\s*"
    r"((?:[a-z0-9-]{1,63}(?:\.|\s*(?:\[dot\]|\(dot\)|\[punto\]|\(punto\)|\s+punto\s+)\s*))+[a-z]{2,24})\b", re.I)
SEPARADOR_PUNTO = re.compile(r"\s*(?:\[dot\]|\(dot\)|\[punto\]|\(punto\)|\s+punto\s+)\s*", re.I)
EMAIL_RUIDO = re.compile(
    r"\.(png|jpe?g|gif|webp|svg|css|js)$|^(u00|x[0-9a-f]{2})|@(sentry[\w.-]*|[\w.-]*wixpress|example|domain|email)\."
    r"(com|org|net|io)$|@[0-9]+x\.|^[0-9a-f]{16,}@|^(tu|su|nombre|correo|usuario)@", re.I)


def normalizar_email(valor):
    """Limpia un correo y descarta los falsos (imágenes, plantillas, servicios técnicos)."""
    email = valor.strip().strip(".").lower().replace("mailto:", "").split("?")[0]
    if EMAIL_RUIDO.search(email) or "@" not in email:
        return None
    local, _, dominio = email.partition("@")
    if not local or "." not in dominio or len(email) > 254 or dominio.startswith("www."):
        return None
    return email if EMAIL_COMPLETO.fullmatch(email) else None


def emails_en(texto, enlaces, html):
    """Correos de 4 fuentes: enlaces mailto, texto visible, texto oculto con [at] y Cloudflare."""
    encontrados = []
    for href, _ in enlaces:
        if href.lower().startswith("mailto:"):
            encontrados += [normalizar_email(p) for p in unquote(href[7:]).split("?")[0].split(",")]
    encontrados += [normalizar_email(m.group(1)) for m in EMAIL_EN_TEXTO.finditer(texto)]
    for m in EMAIL_OCULTO.finditer(texto):
        dominio = SEPARADOR_PUNTO.sub(".", m.group(2)).replace(" ", "")
        encontrados.append(normalizar_email(f"{m.group(1)}@{dominio}"))
    encontrados += [normalizar_email(decodificar_cfemail(c))
                    for c in re.findall(r'data-cfemail="([0-9a-fA-F]+)"', html)]
    return [e for e in dict.fromkeys(encontrados) if e]          # sin vacíos ni repetidos, en orden


# Celulares (multipaís)
def celulares_en(texto, enlaces, pais="Ecuador"):
    """Celulares desde enlaces tel:, enlaces de WhatsApp y texto visible, con las reglas del país."""
    numeros = []
    for href, _ in enlaces:
        if href.lower().startswith("tel:"):
            numeros.append(unquote(href[4:]))
        m = re.search(r"(?:wa\.me/\+?|phone=(?:%2B|\+)?)(\d{9,15})", href, re.I)
        if m:
            numeros.append(m.group(1))
    telefonos = [normalizar_telefono(n, pais) for n in numeros] + telefonos_en_texto(texto, pais)
    return [t for t in dict.fromkeys(telefonos) if t and es_celular(t)]


# Redes sociales
REDES = {
    "facebook": ("facebook.com", "fb.com", "fb.me"),
    "instagram": ("instagram.com", "instagr.am"),
    "tiktok": ("tiktok.com",),
    "x": ("twitter.com", "x.com"),
    "youtube": ("youtube.com", "youtu.be"),
    "linkedin": ("linkedin.com",),
}
# Enlaces de redes que NO son el perfil del negocio (botones de compartir, inicio de sesión, publicaciones...)
RUTAS_NO_PERFIL = re.compile(r"/(sharer|share|intent|dialog|plugins|tr|login|signup|hashtag|search|watch\?v=|embed|"
                             r"p/|reel/|explore|policies|privacy|legal|help|about/|home\.php|posts/|photos/|videos/|"
                             r"events/|groups/|stories/|status/)", re.I)
SEGMENTOS_RESERVADOS = {"recover", "reg", "lite", "watch", "business", "careers", "settings", "l.php", "login",
                        "signup", "help", "legal", "privacy", "policies", "about", "terms", "accounts", "explore",
                        "direct", "developers", "ads", "marketplace", "home", "i", "hashtag", "results", "feed",
                        "share", "sharer", "dialog", "tr", "web", "embed", "plugins"}


def red_de(url):
    """Si la URL es el perfil de un negocio en una red social, devuelve (red, url limpia); si no, None."""
    try:
        p = urlparse(url)
    except ValueError:
        return None
    host = p.netloc.lower().split(":")[0]
    host = host[4:] if host.startswith("www.") else host
    host = host[2:] if host.startswith("m.") else host
    for red, hosts in REDES.items():
        if any(host == h or host.endswith("." + h) for h in hosts):
            ruta = p.path.rstrip("/")
            segmentos = [s for s in ruta.split("/") if s]
            if not segmentos or RUTAS_NO_PERFIL.search(p.path + ("?" + p.query if p.query else "")):
                return None
            if segmentos[0].lower() in SEGMENTOS_RESERVADOS or host.startswith(("l.", "lm.")):
                return None
            if red == "facebook" and segmentos[0].lower() == "profile.php":
                return red, f"https://www.facebook.com/profile.php?{p.query}"
            if red == "facebook" and segmentos[0].lower() == "pages":
                return red, f"https://www.facebook.com/{'/'.join(segmentos[:3])}"
            return red, f"https://{host}/{segmentos[0]}"          # solo el perfil, sin publicaciones
    return None


def es_red_social(url):
    """True si la URL es de cualquier red social (perfil o no): esas páginas no se pueden leer sin iniciar sesión."""
    host = urlparse(url).netloc.lower().split(":")[0]
    return any(host == h or host.endswith("." + h) for hosts in REDES.values() for h in hosts)


def redes_en(enlaces):
    """{'facebook': url, 'instagram': url, ...} con el primer perfil encontrado de cada red."""
    redes = {}
    for href, _ in enlaces:
        r = red_de(href)
        if r and r[0] not in redes:
            redes[r[0]] = r[1]
    return redes


# Páginas de contacto
PISTAS_CONTACTO = re.compile(r"(contacto|cont[aá]ctanos|contact|reserva|booking|ubicaci[oó]n|"
                             r"nosotros|qui[eé]nes[-_ ]somos|about)", re.I)


def mismo_sitio(netloc):
    """'www.hotel.com.ec' ==> 'hotel.com.ec' (para saber si un enlace es del mismo sitio)."""
    partes = netloc.lower().split(":")[0].split(".")
    if len(partes) >= 3 and partes[-2] in ("co", "com", "org", "net", "gob", "edu") and len(partes[-1]) == 2:
        return ".".join(partes[-3:])
    return ".".join(partes[-2:])


def paginas_candidatas(url_base, enlaces, limite):
    """Enlaces del mismo sitio cuyo texto o dirección parece de contacto. Contacto primero, luego reservas, luego 'nosotros'."""
    sitio = mismo_sitio(urlparse(url_base).netloc)
    puntuadas, vistas = [], {url_base.split("#")[0].rstrip("/")}
    for href, texto in enlaces:
        p = urlparse(href)
        destino = href.split("#")[0].rstrip("/")
        if not href.startswith("http") or mismo_sitio(p.netloc) != sitio or destino in vistas:
            continue
        m = PISTAS_CONTACTO.search(f"{p.path} {texto}")
        if not m:
            continue
        palabra = m.group(1).lower()
        puntaje = 0 if palabra.startswith("cont") else (1 if palabra.startswith(("reserva", "booking")) else 2)
        vistas.add(destino)
        puntuadas.append((puntaje, destino))
    return [url for _, url in sorted(puntuadas)[:limite]]


# Tipos de "web" que no son el sitio propio del negocio
# Plataformas de reservas: la página es de la plataforma, no del hotel; nunca muestran el correo del negocio.
PLATAFORMAS = ("booking.com", "beds24.com", "oyorooms.com", "airbnb.", "expedia.", "tripadvisor.", "despegar.",
               "hotels.com", "trivago.", "agoda.", "hostelworld.", "decolar.", "almundo.", "pedidosya.", "rappi.",
               "ubereats.", "ifood.", "didi-food", "opentable.", "thefork.")
# Páginas "link en bio": una sola página con los enlaces del negocio (correo, WhatsApp, Instagram...). Sí se leen.
LINK_EN_BIO = ("linktr.ee", "whats.link", "beacons.ai", "bio.link", "taplink.", "linkin.bio", "lnk.bio")


def tipo_de_web(web):
    """'propia' | 'plataforma' | 'link_en_bio' | 'red_social' | 'whatsapp' | None"""
    if not web:
        return None
    w = web.lower()
    if "wa.me" in w or "whatsapp.com" in w:
        return "whatsapp"
    if es_red_social(web):
        return "red_social"
    if any(p in w for p in PLATAFORMAS):
        return "plataforma"
    if any(p in w for p in LINK_EN_BIO):
        return "link_en_bio"
    return "propia"


# Elegir el mejor correo
PROVEEDORES_GRATIS = ("gmail.", "hotmail.", "outlook.", "yahoo.", "live.", "icloud.", "msn.", "aol.", "proton")
DOMINIOS_TECNICOS = ("support.com", "sentry.", "wixpress.com", "wix.com", "godaddy.", "squarespace.", "hostinger.",
                     "cloudflare.", "googlemail.com", "w3.org", "schema.org")


def elegir_emails(emails, web):
    """Ordena los correos del más al menos confiable y descarta los técnicos:
    1.º mismo dominio que la web del negocio (info@hotelx.com en hotelx.com)
    2.º proveedores gratuitos (gmail, hotmail...), muy usados por negocios pequeños
    3.º otros dominios (suelen ser de la agencia que hizo la web o de terceros)"""
    sitio = mismo_sitio(urlparse(web).netloc) if web else ""
    limpios = [e for e in emails
               if not any(t in e.split("@")[1] for t in DOMINIOS_TECNICOS)
               and not re.match(r"(no-?reply|do-?not-?reply|postmaster|webmaster|abuse)@", e)]

    def puntaje(e):
        dominio = e.split("@")[1]
        if sitio and mismo_sitio(dominio) == sitio:
            return 0
        if dominio.startswith(PROVEEDORES_GRATIS):
            return 1
        return 2
    return sorted(dict.fromkeys(limpios), key=puntaje)


# Lectura de páginas: requests (rápido) y Selenium (segunda opción)
def analizar_html(html, url):
    """Con BeautifulSoup: devuelve (texto visible, enlaces [(href absoluto, texto)])."""
    sopa = BeautifulSoup(html, "html.parser")
    for etiqueta in sopa(["script", "style", "noscript"]):
        etiqueta.decompose()                                       # quitamos código JS/CSS
    texto = sopa.get_text(" ", strip=True)
    enlaces = [(urljoin(url, a["href"].strip()), a.get_text(" ", strip=True))
               for a in sopa.find_all("a", href=True)]
    return texto, enlaces


def leer_pagina(url):
    """Descarga con requests (sin JavaScript). Devuelve (url final, html, texto, enlaces) o None."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
    except requests.RequestException:
        return None                                                # web caída, SSL inválido, tiempo agotado...
    if r.status_code != 200 or "html" not in r.headers.get("Content-Type", "html"):
        return None
    return (r.url, r.text, *analizar_html(r.text, r.url))


def leer_con_selenium(driver, url):
    """Plan B: abre la página en Chrome, que SÍ ejecuta JavaScript (sitios Wix, React...). Mismo formato de salida."""
    try:
        driver.set_page_load_timeout(25)
        driver.get(url)
    except TimeoutException:
        driver.execute_script("window.stop();")                    # usamos lo que alcanzó a cargar
    except Exception:
        return None
    pausa(2, 3)                                                    # tiempo para que el JavaScript pinte el contenido
    html = driver.page_source
    return (driver.current_url, html, *analizar_html(html, driver.current_url))


def recorrer_sitio(web, pais, lector, max_paginas, seguir_enlaces=True):
    """Lee el inicio y las páginas de contacto que encuentre, con el lector indicado (requests o Selenium)."""
    emails, celulares, redes = [], [], {}
    por_visitar, visitadas, cargadas = [web], set(), 0
    while por_visitar and len(visitadas) <= max_paginas:
        url = por_visitar.pop(0)
        if url in visitadas:
            continue
        visitadas.add(url)
        pagina = lector(url)
        if pagina is None:
            continue
        cargadas += 1
        url_final, html, texto, enlaces = pagina
        emails += emails_en(texto, enlaces, html)
        celulares += celulares_en(texto, enlaces + [(url_final, "")], pais)   # whats.link redirige a WhatsApp
        for r, perfil in redes_en(enlaces).items():
            redes.setdefault(r, perfil)

        if len(visitadas) == 1 and seguir_enlaces:                 # desde el inicio: buscar páginas de contacto reales
            candidatas = paginas_candidatas(url_final, enlaces, max_paginas)
            base = f"{urlparse(url_final).scheme}://{urlparse(url_final).netloc}"
            por_visitar += candidatas or [urljoin(base, p) for p in PAGINAS_CONTACTO]   # si no hay, adivinamos
        if emails and celulares:                                   # ya tenemos ambos: no hace falta seguir
            break
        pausa(0.5, 1.5)
    return emails, celulares, redes, len(visitadas), cargadas


def extraer_contactos_web(web, pais="Ecuador", driver=None):
    """Correos, celulares y redes desde la web del negocio.
    1.º requests + BeautifulSoup (rápido). 2.º si no hubo correo y hay navegador: Selenium (plan B)."""
    tipo = tipo_de_web(web)
    resultado = {"emails_web": [], "celulares_web": [], "redes": {}, "paginas_revisadas": 0,
                 "tipo_web": tipo, "metodo_email": None}
    if tipo is None or tipo == "plataforma":                       # sin web, o Booking/OYO...: nada que leer
        return resultado
    if tipo == "whatsapp":                                         # el celular está en la propia URL
        resultado["celulares_web"] = celulares_en("", [(web, "")], pais)
        return resultado
    if tipo == "red_social":                                       # no se lee sin sesión, pero el perfil es un contacto
        red = red_de(web)
        if red:
            resultado["redes"] = {red[0]: red[1]}
        return resultado

    seguir = tipo == "propia"                                      # en un "link en bio" no hay páginas de contacto
    emails, celulares, redes, revisadas, cargadas = recorrer_sitio(web, pais, leer_pagina, MAX_PAGINAS_CONTACTO, seguir)
    metodo = "requests" if emails else None

    # Plan B: la web no cargó con requests, o cargó sin correo (probable sitio hecho con JavaScript)
    if not emails and driver is not None and USAR_SELENIUM_WEB:
        e2, c2, r2, rev2, _ = recorrer_sitio(web, pais, lambda u: leer_con_selenium(driver, u),
                                             MAX_PAGINAS_SELENIUM, seguir)
        emails, celulares, revisadas = e2, celulares + c2, revisadas + rev2
        for r, perfil in r2.items():
            redes.setdefault(r, perfil)
        metodo = "selenium" if emails else None

    resultado.update(emails_web=elegir_emails(emails, web), celulares_web=list(dict.fromkeys(celulares)),
                     redes=redes, paginas_revisadas=revisadas, metodo_email=metodo)
    return resultado


## Paso 10 · Ejecución (Fases 1, 2 y 3)

**Fase 1** guarda **todos** los negocios, sin filtrar. Así se puede calcular el percentil frente a toda la competencia, ver la evolución del rating entre semanas y producir indicadores de mercado para BI.

**Fase 2** — optimización v10: **solo visita los negocios dentro del rango del cliente**. Visitar una ficha tarda 15–25 segundos; filtrar antes evita visitas inútiles (en la v09, 160 de 300 fichas estaban fuera de rango). Además, **reintenta** una vez las fichas que quedaron sin histograma o sin reseñas.

**Fase 3** (sin volver a Maps):

1. **Historial**: compara el rating actual con el de la búsqueda anterior → `variacion_rating` y `tendencia` (`empeora`, `estable`, `mejora`, `sin historial`). Así se responde la pregunta 3 del desafío: el mismo negocio se **actualiza**, no se duplica.
2. **Percentil** frente a toda la población (no solo frente a los visitados).
3. **Regla del cliente** con el rating **más reciente**.
4. **Prioridad** y su **motivo** (`motivo_prioridad`), para que el equipo comercial sepa por qué llamar primero.

Qué se verá en la salida: una línea por búsqueda en la Fase 1, una línea por ficha en la Fase 2 y el resumen de leads al final.

**v11:** la Fase 1 solo busca las ciudades del **bloque activo**, con un descanso largo cada `DESCANSO_CADA` búsquedas. La Fase 2 visita los candidatos pendientes de **todos** los bloques ya ejecutados (peores primero, hasta `LOTE_FICHAS`).


In [11]:
# 4. EJECUCIÓN: Fase 1 → Fase 2  → Fase 3

# FASE 1 · Lista de negocios. Guarda TODOS los negocios encontrados (sin filtrar):
# los necesitamos para el percentil, el historial de ratings y los indicadores de BI.

def fase_1(driver, wait):
    negocios = cargar_csv(SALIDA_NEGOCIOS)
    hechas = set()
    if not negocios.empty:                     # reanudable: saltamos lo que ya se buscó HOY
        hoy = negocios[negocios["fecha_extraccion"] == HOY]
        hechas = set(zip(hoy["descripcion"], hoy["ciudad"], hoy["pais"]))

    busquedas = 0
    print(f"Fase 1 · bloque {BLOQUE_ACTIVO}: {', '.join(PAISES_ACTIVOS)}")
    for desc in DESCRIPCION:
        for ciudad, pais in CIUDADES:          # cada par ("ciudad", "país") se desempaqueta en dos variables
            if (desc, ciudad, pais) in hechas:
                print(f"Saltando {desc} | {ciudad}, {pais} (ya hecho hoy)")
                continue
            print(f"Buscando {desc} | {ciudad}, {pais}...")
            filas = extraer_lista(driver, wait, desc, ciudad, pais)
            if filas:
                negocios = pd.concat([negocios, pd.DataFrame(filas)], ignore_index=True)
                # Una fila por negocio y por día: así se construye el historial del rating (pregunta 3 del desafío)
                negocios = negocios.drop_duplicates(subset=["place_id", "fecha_extraccion"])
                negocios.to_csv(SALIDA_NEGOCIOS, index=False, encoding="utf-8-sig")   # guardado tras cada búsqueda
            print(f"    {len(filas)} negocios | acumulado: {len(negocios)}")
            pausa(5, 12)                       # pausa "humana" entre búsquedas
            busquedas += 1
            if busquedas % DESCANSO_CADA == 0:  # descanso largo cada DESCANSO_CADA búsquedas
                segundos = random.uniform(*DESCANSO_SEG)
                print(f"    Descanso de {segundos:.0f} s para no saturar a Google...")
                time.sleep(segundos)
    return negocios

# Funciones de apoyo para la Fase 2 y la Fase 3
def ultimo_registro(negocios):
    """Una fila por negocio: su dato más reciente (si se buscó varias semanas, la última)."""
    return negocios.sort_values("fecha_extraccion").drop_duplicates("place_id", keep="last")


def en_rango_cliente(df):
    """REGLA DEL CLIENTE: rating entre RATING_MIN y RATING_MAX y más de MIN_RESENAS reseñas.
    Devuelve una columna de True/False (True = el negocio es candidato)."""
    return df["rating"].between(RATING_MIN, RATING_MAX) & (df["n_resenas"] > MIN_RESENAS)


def fichas_por_reintentar(fichas):
    """Fichas visitadas donde Maps no mostró el histograma o ninguna reseña.
    Suele ocurrir al final de lotes largos, cuando Google empieza a limitar la vista."""
    if fichas.empty or not REINTENTAR_FICHAS:
        return set()
    intentos = fichas["intentos"].fillna(1) if "intentos" in fichas else pd.Series(1, index=fichas.index)
    incompletas = fichas["pct_negativas"].isna() | (fichas["resenas_visibles"].fillna(0) == 0)
    return set(fichas.loc[incompletas & (intentos < MAX_INTENTOS), "place_id"])


# FASE 2 · Fichas. OPTIMIZACIÓN v10: solo se visitan los negocios dentro del rango del cliente.
# Visitar una ficha tarda 15-25 s: filtrar ANTES de visitar es lo que ahorra tiempo.
def fase_2_fichas(driver, wait, negocios):
    fichas = cargar_csv(SALIDA_FICHAS)
    resenas = cargar_csv(SALIDA_RESENAS)
    hechos = set(fichas["place_id"]) if not fichas.empty else set()
    reintentar = fichas_por_reintentar(fichas)

    ultimo = ultimo_registro(negocios)
    candidatos = ultimo[en_rango_cliente(ultimo)]                         # ← el filtro del cliente
    nuevos = candidatos[~candidatos["place_id"].isin(hechos)]
    repetir = candidatos[candidatos["place_id"].isin(reintentar)]
    lote = pd.concat([nuevos, repetir]).sort_values("rating").head(LOTE_FICHAS)   # peores primero

    print(f"\nNegocios en la base: {len(ultimo)} | en el rango del cliente "
          f"({RATING_MIN}–{RATING_MAX}★ y > {MIN_RESENAS} reseñas): {len(candidatos)}")
    print(f"Fichas ya hechas: {len(hechos)} | nuevas por visitar: {len(nuevos)} | "
          f"a reintentar: {len(repetir)} | este lote: {len(lote)}")

    for i, (_, neg) in enumerate(lote.iterrows(), start=1):
        pid = neg["place_id"]
        es_reintento = pid in hechos
        try:
            detalle = extraer_detalle(driver, wait, neg["url"], neg["pais"])       # teléfono, web, dirección
            resumen, visibles = resumen_resenas(driver, wait)                     # histograma, temas, reseñas
            contactos = extraer_contactos_web(detalle["web"], neg["pais"], driver)   # correo, celular, redes
        except Exception as e:
            print(f"    ({i}) Error con {neg['nombre']}: {str(e)[:80]}")
            continue

        # Celular final: el de Google Maps si es celular; si no, el primero hallado en la web
        celular = detalle["telefono"] if es_celular(detalle["telefono"]) else None
        celular = celular or next(iter(contactos["celulares_web"]), None)
        email = next(iter(contactos["emails_web"]), None)

        previos = 0
        if es_reintento:
            fila_vieja = fichas.loc[fichas["place_id"] == pid]
            previos = int(fila_vieja["intentos"].fillna(1).iloc[0]) if "intentos" in fichas else 1
            fichas = fichas[fichas["place_id"] != pid]                    # reemplazamos la ficha incompleta
            if not resenas.empty:
                resenas = resenas[resenas["place_id"] != pid]             # y sus reseñas anteriores

        ficha = {**neg.to_dict(), **detalle, **resumen,
                 "celular": celular, "email": email,
                 "emails_web": "; ".join(contactos["emails_web"]),
                 "celulares_web": "; ".join(contactos["celulares_web"]),
                 **{red: contactos["redes"].get(red) for red in REDES},      # facebook, instagram, tiktok...
                 "paginas_revisadas": contactos["paginas_revisadas"],
                 "tipo_web": contactos["tipo_web"], "metodo_email": contactos["metodo_email"],
                 "intentos": previos + 1, "fecha_ficha": HOY}
        fichas = pd.concat([fichas, pd.DataFrame([ficha])], ignore_index=True)
        fichas.to_csv(SALIDA_FICHAS, index=False, encoding="utf-8-sig")       # guardado por ficha

        if visibles:   # tabla larga: una fila por reseña
            filas = [{"place_id": pid, "nombre": neg["nombre"], "ciudad": neg["ciudad"], "pais": neg["pais"],
                      **r, "fecha_extraccion": HOY} for r in visibles]
            resenas = pd.concat([resenas, pd.DataFrame(filas)], ignore_index=True)
        if visibles or es_reintento:
            resenas.to_csv(SALIDA_RESENAS, index=False, encoding="utf-8-sig")

        pct = resumen["pct_negativas"]
        print(f"    ({i}/{len(lote)}) {'[reintento] ' if es_reintento else ''}{neg['nombre']} ({neg['pais']}) | "
              f"{neg['rating']}★ | tel: {detalle['telefono']} | email: {email} ({contactos['metodo_email'] or '—'}) | "
              f"% 1-2★: {'—' if pct is None else f'{pct:.0%}'} | visibles: {resumen['resenas_visibles']}")
        pausa(3, 6)
    return fichas


# FASE 3 · Clasificación (sin volver a Google Maps)
def historial_rating(negocios):
    """Por negocio: rating actual (última búsqueda), rating anterior (búsqueda previa) y su variación.
    Con una sola fecha de búsqueda, la tendencia es 'sin historial'."""
    orden = negocios.sort_values(["place_id", "fecha_extraccion"])
    actual = orden.drop_duplicates("place_id", keep="last").set_index("place_id")
    previo = orden[orden.duplicated("place_id", keep="last")].drop_duplicates("place_id", keep="last").set_index("place_id")

    h = pd.DataFrame({"rating_actual": actual["rating"], "n_resenas_actual": actual["n_resenas"],
                      "fecha_actual": actual["fecha_extraccion"]})
    h["rating_anterior"] = previo["rating"]
    h["fecha_anterior"] = previo["fecha_extraccion"]
    h["variacion_rating"] = (h["rating_actual"] - h["rating_anterior"]).round(1)

    def tendencia(v):
        if pd.isna(v):
            return "sin historial"
        if v <= -UMBRAL_TENDENCIA:
            return "empeora"
        if v >= UMBRAL_TENDENCIA:
            return "mejora"
        return "estable"
    h["tendencia"] = h["variacion_rating"].apply(tendencia)
    return h


def percentil_poblacion(negocios):
    """Posición de cada negocio frente a TODA su competencia (mismo rubro, país y ciudad):
    0.05 = está entre el 5% peor calificado. Se calcula sobre todos los negocios, no solo los visitados."""
    u = ultimo_registro(negocios)
    u = u[u["n_resenas"] > MIN_RESENAS]
    rango = u.groupby(["descripcion", "pais", "ciudad"])["rating"].rank(pct=True, method="max")
    return pd.Series(rango.values, index=u["place_id"].values, name="percentil").round(3)


def prioridad(r):
    """Devuelve (prioridad, motivo). Alta si cumple AL MENOS una señal fuerte; si no, Media."""
    motivos = []
    if pd.notna(r["pct_negativas"]) and r["pct_negativas"] >= PCT_NEG_ALTA:
        motivos.append(f"{r['pct_negativas']:.0%} de reseñas 1-2★")
    if r["rating"] <= RATING_ALTA:
        motivos.append(f"rating {r['rating']:.1f} ≤ {RATING_ALTA}")
    if pd.notna(r["variacion_rating"]) and r["variacion_rating"] <= -CAIDA_ALERTA:
        motivos.append(f"rating cayó {abs(r['variacion_rating']):.1f}")
    return ("Alta", "; ".join(motivos)) if motivos else ("Media", "en rango del cliente")


COLUMNAS_PRIMERO = ["prioridad", "motivo_prioridad", "nombre", "descripcion", "pais", "ciudad", "rating",
                    "n_resenas", "pct_negativas", "percentil", "tendencia", "variacion_rating",
                    "telefono", "tipo_telefono", "celular", "email", "web", "tipo_web", "facebook", "instagram",
                    "temas", "resenas_visibles", "negativas_visibles"]


def construir_leads(fichas, negocios):
    """Une fichas + historial + percentil, aplica la regla del cliente con el rating MÁS RECIENTE
    y asigna la prioridad. Guarda leads_11.csv con las columnas clave primero."""
    f = fichas.drop(columns=[c for c in ["percentil", "prioridad", "motivo_prioridad"] if c in fichas])
    f = f.join(historial_rating(negocios), on="place_id")
    f["rating_en_ficha"] = f["rating"]                              # rating el día que se visitó la ficha
    f["rating"] = f["rating_actual"].fillna(f["rating"])           # se usa el más reciente
    f["n_resenas"] = f["n_resenas_actual"].fillna(f["n_resenas"])
    f = f.join(percentil_poblacion(negocios), on="place_id")

    dentro = en_rango_cliente(f)
    print(f"\nFichas visitadas: {len(f)} | dentro del rango del cliente hoy: {dentro.sum()} | "
          f"fuera de rango (su rating cambió o se visitaron en versiones anteriores): {(~dentro).sum()}")
    leads = f[dentro].copy()
    leads[["prioridad", "motivo_prioridad"]] = leads.apply(lambda r: pd.Series(prioridad(r)), axis=1)
    # Mismo correo en varios negocios (cadenas): un solo contacto comercial para todos
    leads["email_compartido"] = leads["email"].map(leads["email"].value_counts()).fillna(0).gt(1)

    leads["_orden"] = leads["prioridad"].map({"Alta": 0, "Media": 1})
    leads = leads.sort_values(["_orden", "pct_negativas"], ascending=[True, False]).drop(columns="_orden")
    primero = [c for c in COLUMNAS_PRIMERO if c in leads]
    leads = leads[primero + [c for c in leads.columns if c not in primero]]
    leads.to_csv(SALIDA_LEADS, index=False, encoding="utf-8-sig")
    print("Leads por prioridad:", leads["prioridad"].value_counts().to_dict())
    print(f"Leads guardados: {len(leads)} en {SALIDA_LEADS}")
    return leads


# PROGRAMA PRINCIPAL
if __name__ == "__main__":
    negocios, fichas = pd.DataFrame(), pd.DataFrame()
    driver = iniciar_driver()
    wait = WebDriverWait(driver, 15)
    try:
        negocios = fase_1(driver, wait)
        if not negocios.empty:
            fichas = fase_2_fichas(driver, wait, negocios)
    finally:
        driver.quit()   # se cierra Chrome aunque haya error o Ctrl+C
        print("\nNavegador cerrado.")
    if not fichas.empty:
        construir_leads(fichas, negocios)


Fase 1 · bloque 1: México
Buscando HOTELES | Ciudad de México, México...
    40 negocios | acumulado: 35
Buscando HOTELES | Guadalajara, México...
    60 negocios | acumulado: 91
Buscando RESTAURANTES | Ciudad de México, México...
    112 negocios | acumulado: 195
Buscando RESTAURANTES | Guadalajara, México...
    61 negocios | acumulado: 250

Negocios en la base: 250 | en el rango del cliente (1.5–3.5★ y > 10 reseñas): 14
Fichas ya hechas: 0 | nuevas por visitar: 14 | a reintentar: 0 | este lote: 14
    (1/14) Gran Hotel Galerías (México) | 2.9★ | tel: +526645751886 | email: None (—) | % 1-2★: 47% | visibles: 0
    (2/14) Hotel Virreyes México (México) | 2.9★ | tel: +525513991682 | email: reservaciones@virreyesmexico.com (requests) | % 1-2★: 50% | visibles: 5
    (3/14) Alameda House (México) | 3.0★ | tel: +525582427804 | email: None (—) | % 1-2★: 42% | visibles: 5
    (4/14) Hotel Brazil (México) | 3.1★ | tel: +523346242912 | email: None (—) | % 1-2★: 36% | visibles: 5
    (5/14) Hot

## Paso 11 · Revisión de resultados

Se puede ejecutar **sola** (sin scraping) para ver el estado de los archivos: tamaño de la base, fichas completas, leads por prioridad, cobertura de contacto, material disponible para la IA y tendencia del rating.


In [12]:
# 5. REVISIÓN DE RESULTADOS
negocios = cargar_csv(SALIDA_NEGOCIOS)
fichas = cargar_csv(SALIDA_FICHAS)
leads = cargar_csv(SALIDA_LEADS)
resenas = cargar_csv(SALIDA_RESENAS)

if not negocios.empty:
    ultimo = ultimo_registro(negocios)
    print(f"BASE      → negocios: {len(ultimo)} | búsquedas (fechas): {negocios['fecha_extraccion'].nunique()} | "
          f"en rango del cliente: {en_rango_cliente(ultimo).sum()}")
if not fichas.empty:
    con_red = fichas[[r for r in REDES if r in fichas]].notna().any(axis=1)
    print(f"FICHAS    → visitadas: {len(fichas)} | con histograma: {fichas['pct_negativas'].notna().sum()} | "
          f"con reseñas visibles: {(fichas['resenas_visibles'].fillna(0) > 0).sum()} | reseñas guardadas: {len(resenas)}")
if not leads.empty:
    con_red = leads[[r for r in REDES if r in leads]].notna().any(axis=1)
    canal = leads["telefono"].notna() | leads["email"].notna() | con_red
    print(f"LEADS     → total: {len(leads)} | por prioridad: {leads['prioridad'].value_counts().to_dict()}")
    print(f"CONTACTO  → teléfono: {leads['telefono'].notna().sum()} | celular: {leads['celular'].notna().sum()} | "
          f"correo: {leads['email'].notna().sum()} | red social: {con_red.sum()} | algún canal: {canal.sum()} "
          f"({canal.mean():.0%})")
    print(f"IA        → leads con al menos 1 reseña negativa con texto: {(leads['negativas_visibles'].fillna(0) > 0).sum()} | "
          f"con temas de Google: {(leads['temas'].fillna('') != '').sum()}")
    print("TENDENCIA →", leads["tendencia"].value_counts().to_dict())
    print("POR PAÍS  →", leads.groupby(["pais", "descripcion"]).size().to_dict())
    display(leads[[c for c in COLUMNAS_PRIMERO if c in leads]].head(20))


BASE      → negocios: 250 | búsquedas (fechas): 1 | en rango del cliente: 14
FICHAS    → visitadas: 14 | con histograma: 14 | con reseñas visibles: 11 | reseñas guardadas: 55
LEADS     → total: 14 | por prioridad: {'Media': 10, 'Alta': 4}
CONTACTO  → teléfono: 14 | celular: 14 | correo: 5 | red social: 2 | algún canal: 14 (100%)
IA        → leads con al menos 1 reseña negativa con texto: 11 | con temas de Google: 4
TENDENCIA → {'sin historial': 14}
POR PAÍS  → {('México', 'HOTELES'): 14}


,prioridad,motivo_prioridad,nombre,descripcion,pais,ciudad,rating,n_resenas,pct_negativas,percentil,...,tipo_telefono,celular,email,web,tipo_web,facebook,instagram,temas,resenas_visibles,negativas_visibles
0,Alta,50% de reseñas 1-2★; rating 2.9 ≤ 3.0,Hotel Virreyes México,HOTELES,México,Ciudad de México,2.9,22,0.500,0.029,...,celular_o_fijo,+525513991682,reservaciones@virreyesmexico.com,https://www.virreyesmexico.com/,propia,NaN,NaN,ruido: 3; chinches: 2; habitación: 2; cliente:...,5,3
1,Alta,47% de reseñas 1-2★; rating 2.9 ≤ 3.0,Gran Hotel Galerías,HOTELES,México,Guadalajara,2.9,166,0.470,0.018,...,celular_o_fijo,+526645751886,NaN,NaN,NaN,NaN,NaN,fotos: 9; económico: 6; precio: 6; sábanas: 5;...,0,0
2,Alta,42% de reseñas 1-2★; rating 3.0 ≤ 3.0,Alameda House,HOTELES,México,Ciudad de México,3.0,650,0.417,0.057,...,celular_o_fijo,+525582427804,NaN,http://alamedahouse.mx/,propia,NaN,NaN,NaN,5,3
3,Alta,42% de reseñas 1-2★,HOSTAL CONDESA,HOTELES,México,Ciudad de México,3.3,36,0.417,0.143,...,celular_o_fijo,+523342491881,NaN,NaN,NaN,NaN,NaN,NaN,5,3
4,Media,en rango del cliente,Hotel Brazil,HOTELES,México,Guadalajara,3.1,759,0.362,0.036,...,celular_o_fijo,+523346242912,NaN,https://www.oyorooms.com/MX/105476/,plataforma,NaN,NaN,NaN,5,4
5,Media,en rango del cliente,Hotel Mallorca,HOTELES,México,Ciudad de México,3.1,2508,0.349,0.086,...,celular_o_fijo,+525555664833,reservaciones@hmallorca.com.mx,http://www.hmallorca.com.mx/,propia,NaN,NaN,NaN,5,3
6,Media,en rango del cliente,"Hotel Florida, Centro Histórico",HOTELES,México,Ciudad de México,3.2,1142,0.308,0.114,...,celular_o_fijo,+525583054760,hotelfloridacdmx@gmail.com,https://hotelfloridacdmx.com.mx/,propia,https://web.facebook.com/HotelFloridacdmx,NaN,NaN,5,2
7,Media,en rango del cliente,"Hotel Canada, Guadalajara",HOTELES,México,Guadalajara,3.3,604,0.293,0.054,...,celular_o_fijo,+523336194747,hotelcanadagdl@outlook.com,https://hotelcanada-gdl.com/,propia,NaN,NaN,NaN,5,5
8,Media,en rango del cliente,Rooms Zocalo,HOTELES,México,Ciudad de México,3.5,132,0.288,0.257,...,celular_o_fijo,+525555428722,NaN,NaN,NaN,NaN,NaN,ruido: 10; dormir: 8; piso: 8; regaderas: 7; l...,0,0
9,Media,en rango del cliente,HOTEL METROPOLITAN,HOTELES,México,Guadalajara,3.5,592,0.279,0.089,...,celular_o_fijo,+523336149382,NaN,https://hotelmetropolitanguadalajara.top/,propia,NaN,NaN,NaN,5,2


## Prueba rápida de contactos (sin abrir Google Maps)

Ejecutar las celdas de los pasos 1 a 9 y luego esta. Pegar en la lista las webs a probar (por ejemplo, hoteles cuyo correo se conoce).

In [13]:
PAIS_PRUEBA = "Ecuador"     # país de las webs de prueba (define cómo se leen los teléfonos)
WEBS_PRUEBA = [
    # "https://www.ejemplo-hotel.com",
]
driver_prueba = iniciar_driver() if (WEBS_PRUEBA and USAR_SELENIUM_WEB) else None   # Chrome solo para el plan B
try:
    for web in WEBS_PRUEBA:
        r = extraer_contactos_web(web, PAIS_PRUEBA, driver_prueba)
        print(f"{web}  [{r['tipo_web']}]\n   correos: {r['emails_web']} (método: {r['metodo_email']})\n"
              f"   celulares: {r['celulares_web']}\n   redes: {r['redes']}\n   páginas revisadas: {r['paginas_revisadas']}\n")
finally:
    if driver_prueba:
        driver_prueba.quit()
